In [2]:
import numpy as np
import pandas as pd
import tensorflow 
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense,Embedding,LSTM,TextVectorization

In [3]:
jokes = pd.read_csv("/kaggle/input/datasets/thedevastator/one-million-reddit-jokes/one-million-reddit-jokes.csv")

In [4]:
jokes.head()

,type,id,subreddit.id,subreddit.name,subreddit.nsfw,created_utc,permalink,domain,url,selftext,title,score
0,post,ftbp1i,2qh72,jokes,False,1585785543,https://old.reddit.com/r/Jokes/comments/ftbp1i...,self.jokes,NaN,My corona is covered with foreskin so it is no...,I am soooo glad I'm not circumcised!,2
1,post,ftboup,2qh72,jokes,False,1585785522,https://old.reddit.com/r/Jokes/comments/ftboup...,self.jokes,NaN,It's called Google Sheets.,Did you know Google now has a platform for rec...,9
2,post,ftbopj,2qh72,jokes,False,1585785508,https://old.reddit.com/r/Jokes/comments/ftbopj...,self.jokes,NaN,The vacuum doesn't snore after sex.\r\n\r\n&am...,What is the difference between my wife and my ...,15
3,post,ftbnxh,2qh72,jokes,False,1585785428,https://old.reddit.com/r/Jokes/comments/ftbnxh...,self.jokes,NaN,[removed],My last joke for now.,9
4,post,ftbjpg,2qh72,jokes,False,1585785009,https://old.reddit.com/r/Jokes/comments/ftbjpg...,self.jokes,NaN,[removed],The Nintendo 64 turns 18 this week...,134


In [5]:
jokes = jokes.iloc[:,-3]

In [6]:
jokes

0         My corona is covered with foreskin so it is no...
1                                It's called Google Sheets.
2         The vacuum doesn't snore after sex.\r\n\r\n&am...
3                                                 [removed]
4                                                 [removed]
                                ...                        
999993    *zyan malik or whatever leaves 1d.  \r\n*Kayne...
999994                                            [deleted]
999995                                         I'll be Bach
999996    So a moth goes into a podiatrists office.\r\n\...
999997                                            [deleted]
Name: selftext, Length: 999998, dtype: object

In [7]:
type(jokes)

pandas.core.series.Series

In [8]:
jokes[0]

'My corona is covered with foreskin so it is not exposed to viruses.'

In [9]:
jokes.isnull().sum()

np.int64(4515)

In [10]:
jokes = jokes.dropna()

In [11]:
jokes.shape

(995483,)

In [12]:
temp = [len(s.split(' ')) for s in jokes]

In [13]:
max(temp)

23719

In [14]:
jokes = [joke for joke in jokes if (len(str(joke).split()) <= 30 and joke != '[removed]' and joke != '[deleted]')]

In [15]:
len(jokes)

461173

In [16]:
# training on this huge data is taking too much time 
# lets move with 2000 input rows

jokes = jokes[:2000]

In [17]:
len(jokes)

2000

In [18]:
# i have extracted the joke senteneces that are upto 30 words long
# now i will tokenize the data and form ngrams
# now build ngrams

vectorizer = TextVectorization(ragged = False)
vectorizer.adapt(jokes)

input_sentences = []
for s in jokes:
    token = vectorizer(s)
    for i in range(1,len(token)):
        input_sentences.append(token[:i+1])

2026-07-06 16:24:15.860454: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [40]:
print(input_sentences[0])
print(input_sentences[1])

[  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0  15 195]
[  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0  15 195  14]


In [20]:
input_sentences[1]

<tf.Tensor: shape=(3,), dtype=int64, numpy=array([ 15, 195,  14])>

In [21]:
from keras.utils import pad_sequences
input_sentences = pad_sequences(input_sentences,padding='pre')

In [22]:
X = input_sentences[:,:-1]
Y = input_sentences[:,-1]

In [23]:
X

array([[   0,    0,    0, ...,    0,    0,   15],
       [   0,    0,    0, ...,    0,   15,  195],
       [   0,    0,    0, ...,   15,  195,   14],
       ...,
       [   0,    0,    0, ...,   12,   13,    3],
       [   0,    0,    0, ...,   13,    3,  363],
       [   0,    0,    0, ...,    3,  363, 1133]],
      shape=(14884, 29), dtype=int32)

In [24]:
Y

array([ 195,   14,  794, ...,  363, 1133,  534],
      shape=(14884,), dtype=int32)

In [25]:
vectorizer.vocabulary_size()

3961

In [26]:
Y.shape

(14884,)

In [27]:
X.shape

(14884, 29)

In [28]:
# we are dong categorical supervised learning so conert the output column to categorical data
from keras.utils import to_categorical
Y = to_categorical(Y,num_classes=3962)

In [29]:
Y

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(14884, 3962))

In [30]:
Y.shape

(14884, 3962)

In [31]:
Y[0].argmax()

np.int64(195)

In [42]:
print(Y[0][195])

1.0


In [32]:
model = Sequential()
model.add(Embedding(3961, 100))
model.add(LSTM(150))
#model.add(LSTM(150))
model.add(Dense(3962, activation='softmax'))

In [33]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [34]:
model.compile(loss='categorical_crossentropy', optimizer='adam',metrics=['accuracy'])

In [35]:
print(len(X))
print(len(Y))

14884
14884


In [36]:
min(len(x) for x in X)

29

In [37]:
min([len(y) for y in Y])

3962

In [38]:
model.fit(X,Y,epochs=25,validation_split=0.2)

Epoch 1/25
373/373 ━━━━━━━━━━━━━━━━━━━━ 18s 42ms/step - accuracy: 0.0354 - loss: 7.1475 - val_accuracy: 0.0390 - val_loss: 7.0312
Epoch 2/25
373/373 ━━━━━━━━━━━━━━━━━━━━ 15s 41ms/step - accuracy: 0.0393 - loss: 6.6305 - val_accuracy: 0.0440 - val_loss: 7.1284
Epoch 3/25
 57/373 ━━━━━━━━━━━━━━━━━━━━ 11s 36ms/step - accuracy: 0.0500 - loss: 6.3918

KeyboardInterrupt: 

In [ ]:
# predicting next word (only 1 word)

sentence = 'Why'
vocab = vectorizer.get_vocabulary()

prediction_token = vectorizer(sentence)
prediction_token = pad_sequences([prediction_token],padding='pre',maxlen=29)
predicted = model.predict(prediction_token)
pos = np.argmax(predicted)
print(vocab[pos])

In [ ]:
# predicting next n word 

sentence = 'This is humurous'
n=10
vocab = vectorizer.get_vocabulary()
for i in range(n):
    prediction_token = vectorizer(sentence)
    prediction_token = pad_sequences([prediction_token],padding='pre',maxlen=29)
    predicted = model.predict(prediction_token)
    pos = np.argmax(predicted)
    predictedWord = vocab[pos]

    sentence = sentence + ' ' + predictedWord
    print(sentence)

In [ ]:
# getting top 3 possiblities of next word (single next word)
import heapq

def get_top_three_indices(data_list):
    # Use enumerate to pair each element with its index, then find the 3 largest based on the value
    return heapq.nlargest(3, range(len(data_list)), key=data_list.__getitem__)

sentence = 'thats why i'

prediction_token = vectorizer(sentence)
prediction_token = pad_sequences([prediction_token],padding='pre',maxlen=29)
predicted = model.predict(prediction_token)
pos = get_top_three_indices(predicted.flatten())
print(pos)
for p in pos:
    print(sentence + ' ' + vocab[p]) 